In [ ]:
# ============================================================
# REPRODUCIBLE DATA SETUP — GITHUB
# ============================================================
# This notebook automatically retrieves the thesis repository in Google Colab.
# No manual upload of NASA C-MAPSS FD001 files is required.

import os
import subprocess

REPO_URL = "https://github.com/dilawar1791/turbofan-rul-shap-thesis.git"
REPO_DIR = "/content/turbofan-rul-shap-thesis"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

DATA_DIR = os.path.join(REPO_DIR, "data")
TRAIN_PATH = os.path.join(DATA_DIR, "train_FD001.txt")
TEST_PATH = os.path.join(DATA_DIR, "test_FD001.txt")
RUL_PATH = os.path.join(DATA_DIR, "RUL_FD001.txt")

required_data_files = [TRAIN_PATH, TEST_PATH, RUL_PATH]
missing_data_files = [p for p in required_data_files if not os.path.exists(p)]

if missing_data_files:
    raise FileNotFoundError(
        "Missing required NASA C-MAPSS FD001 files:\n"
        + "\n".join(f" - {p}" for p in missing_data_files)
    )

print("✓ All required FD001 data files found.")
print("Training data :", TRAIN_PATH)
print("Test data     :", TEST_PATH)
print("RUL labels    :", RUL_PATH)

In [ ]:
# ============================================================
# THESIS REVISION EXPERIMENTS
# NASA C-MAPSS FD001
# Professor Feedback — Feature Informativeness Analysis
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. COLUMN NAMES
# ------------------------------------------------------------

columns = (
    ['engine_id', 'cycle'] +
    ['op_setting_1', 'op_setting_2', 'op_setting_3'] +
    [f'sensor_{i}' for i in range(1, 22)]
)

# ------------------------------------------------------------
# 2. LOAD FD001 TRAINING DATA
# ------------------------------------------------------------

train = pd.read_csv(
    TRAIN_PATH,
    sep=r"\s+",
    header=None,
    names=columns
)

print("Dataset shape:", train.shape)
print("Number of engines:", train["engine_id"].nunique())

# ------------------------------------------------------------
# 3. CALCULATE RUL — SAME METHOD AS ORIGINAL NOTEBOOK
# ------------------------------------------------------------

max_cycle = (
    train.groupby("engine_id")["cycle"]
    .max()
    .reset_index()
)

max_cycle.columns = ["engine_id", "max_cycle"]

train = train.merge(
    max_cycle,
    on="engine_id",
    how="left"
)

train["RUL_uncapped"] = (
    train["max_cycle"] - train["cycle"]
)

# Cap RUL at 125 cycles
train["RUL"] = train["RUL_uncapped"].clip(upper=125)

print("\nMaximum uncapped RUL:", train["RUL_uncapped"].max())
print("\nCapped RUL summary:")
print(train["RUL"].describe())

# ------------------------------------------------------------
# 4. DEFINE ORIGINAL 24 FEATURES
# ------------------------------------------------------------

feature_cols = (
    ['op_setting_1', 'op_setting_2', 'op_setting_3'] +
    [f'sensor_{i}' for i in range(1, 22)]
)

print("\nNumber of original predictors:", len(feature_cols))
print(feature_cols)

# ------------------------------------------------------------
# 5. REPRODUCE ORIGINAL ENGINE-LEVEL SPLIT
# ------------------------------------------------------------

engine_ids = train["engine_id"].unique()

train_ids, test_ids = train_test_split(
    engine_ids,
    test_size=0.20,
    random_state=42
)

train_data = train[
    train["engine_id"].isin(train_ids)
].copy()

test_data = train[
    train["engine_id"].isin(test_ids)
].copy()

print("\nTrain engines:", len(train_ids))
print("Test engines :", len(test_ids))

print("Train rows:", len(train_data))
print("Test rows :", len(test_data))

print(
    "Engine overlap:",
    len(set(train_ids).intersection(set(test_ids)))
)

In [ ]:
# ============================================================
# FEATURE INFORMATIVENESS DIAGNOSTIC
# IMPORTANT: calculated on TRAINING DATA ONLY
# ============================================================

feature_diagnostics = []

for feature in feature_cols:

    x = train_data[feature]

    # Basic statistics
    n_unique = x.nunique()
    variance = x.var(ddof=1)
    std_dev = x.std(ddof=1)
    minimum = x.min()
    maximum = x.max()
    value_range = maximum - minimum

    # Fraction of observations occupied by the most common value
    most_common_fraction = x.value_counts(
        normalize=True,
        dropna=False
    ).iloc[0]

    # Correlation with RUL
    pearson_corr = x.corr(train_data["RUL"], method="pearson")
    spearman_corr = x.corr(train_data["RUL"], method="spearman")

    feature_diagnostics.append({
        "feature": feature,
        "unique_values": n_unique,
        "variance": variance,
        "std_dev": std_dev,
        "min": minimum,
        "max": maximum,
        "range": value_range,
        "most_common_fraction": most_common_fraction,
        "pearson_RUL": pearson_corr,
        "spearman_RUL": spearman_corr
    })

feature_diagnostics = pd.DataFrame(feature_diagnostics)

# Sort from lowest variance to highest variance
feature_diagnostics = feature_diagnostics.sort_values(
    by="variance",
    ascending=True
).reset_index(drop=True)

# Improve display
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("FEATURE DIAGNOSTICS — TRAINING DATA ONLY")
print("=" * 120)

print(
    feature_diagnostics.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

# ------------------------------------------------------------
# Explicitly identify exactly constant features
# ------------------------------------------------------------

constant_features = feature_diagnostics.loc[
    feature_diagnostics["unique_values"] <= 1,
    "feature"
].tolist()

print("\n" + "=" * 120)
print("EXACTLY CONSTANT FEATURES")
print("=" * 120)

if constant_features:
    print(constant_features)
else:
    print("No exactly constant features found.")

# ------------------------------------------------------------
# Show features where one value occupies >= 95% of observations
# This is DIAGNOSTIC ONLY — not yet our removal criterion.
# ------------------------------------------------------------

highly_concentrated_features = feature_diagnostics.loc[
    feature_diagnostics["most_common_fraction"] >= 0.95,
    ["feature", "unique_values", "variance", "most_common_fraction"]
]

print("\n" + "=" * 120)
print("FEATURES WITH >=95% OF OBSERVATIONS AT ONE VALUE")
print("Diagnostic only — no features are being removed yet.")
print("=" * 120)

print(
    highly_concentrated_features.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

In [ ]:
# ============================================================
# REMOVE EXACTLY CONSTANT FEATURES
# Criterion: zero variance / one unique value in TRAINING DATA
# ============================================================

constant_features = [
    feature for feature in feature_cols
    if train_data[feature].nunique() == 1
]

informative_features = [
    feature for feature in feature_cols
    if feature not in constant_features
]

print("Original number of predictors:", len(feature_cols))
print("Removed constant predictors:", len(constant_features))
print("Remaining predictors:", len(informative_features))

print("\nRemoved features:")
for feature in constant_features:
    print(" -", feature)

print("\nRetained features:")
for feature in informative_features:
    print(" -", feature)

In [ ]:
# ============================================================
# NEAR-CONSTANT FEATURE DIAGNOSTIC
# For the 17 predictors retained after zero-variance filtering
# ============================================================

near_constant_results = []

for feature in informative_features:
    x = train_data[feature]

    value_counts = x.value_counts(normalize=True, dropna=False)

    dominant_fraction = value_counts.iloc[0]
    second_fraction = value_counts.iloc[1] if len(value_counts) > 1 else 0

    near_constant_results.append({
        "feature": feature,
        "unique_values": x.nunique(),
        "dominant_value": x.value_counts().index[0],
        "dominant_fraction": dominant_fraction,
        "dominant_percent": dominant_fraction * 100,
        "second_value_fraction": second_fraction,
        "variance": x.var(ddof=1),
        "std_dev": x.std(ddof=1),
        "abs_spearman_RUL": abs(
            x.corr(train_data["RUL"], method="spearman")
        )
    })

near_constant_df = pd.DataFrame(near_constant_results)

near_constant_df = near_constant_df.sort_values(
    "dominant_fraction",
    ascending=False
).reset_index(drop=True)

print("NEAR-CONSTANT FEATURE DIAGNOSTIC")
print("=" * 110)

print(
    near_constant_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nPotential near-constant features (>= 95% same value):")

potential_near_constant = near_constant_df[
    near_constant_df["dominant_fraction"] >= 0.95
]

if len(potential_near_constant) == 0:
    print("None")
else:
    print(
        potential_near_constant[
            [
                "feature",
                "unique_values",
                "dominant_value",
                "dominant_percent",
                "abs_spearman_RUL"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

In [ ]:
# ============================================================
# FINAL FEATURE SET AFTER INFORMATIVE-FEATURE SCREENING
# ============================================================

removed_features = [
    "op_setting_3",
    "sensor_1",
    "sensor_5",
    "sensor_6",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

final_features = [
    f for f in feature_cols
    if f not in removed_features
]

print("Original predictors:", len(feature_cols))
print("Removed predictors :", len(removed_features))
print("Retained predictors:", len(final_features))

print("\nRemoved:")
for f in removed_features:
    print(" -", f)

print("\nFinal retained features:")
for f in final_features:
    print(" -", f)

# Safety checks
assert len(final_features) == 16
assert not set(final_features).intersection(removed_features)

X_train = train_data[final_features].copy()
y_train = train_data["RUL"].copy()

X_test = test_data[final_features].copy()
y_test = test_data["RUL"].copy()

print("\nTraining matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

In [ ]:
# ============================================================
# RETRAIN LR, RF AND XGBOOST
# Final 16-feature dataset
# Fixed engine split: random_state = 42
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import numpy as np
import pandas as pd
import time

# ------------------------------------------------------------
# 1. Define authoritative model configurations
# ------------------------------------------------------------

lr_model = LinearRegression()

rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=15,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

models = {
    "Linear Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

# ------------------------------------------------------------
# 2. Train and evaluate
# ------------------------------------------------------------

results = []

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")
    start = time.time()

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    elapsed = time.time() - start

    results.append({
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Training_and_Prediction_Time_sec": elapsed
    })

    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"R²   : {r2:.4f}")
    print(f"Time : {elapsed:.2f} seconds")

# ------------------------------------------------------------
# 3. Results table
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("REVISED MODEL PERFORMANCE — 16 FEATURES")
print("=" * 75)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

In [ ]:
# ============================================================
# SHAP ANALYSIS — REVISED 16-FEATURE MODELS
# Linear Regression, Random Forest, XGBoost
# ============================================================

!pip -q install shap

import shap
import numpy as np
import pandas as pd

# Use a fixed sample for reproducibility and computational efficiency
SHAP_SAMPLE_SIZE = min(1000, len(X_test))

X_shap = X_test.sample(
    n=SHAP_SAMPLE_SIZE,
    random_state=42
)

shap_rankings = {}
shap_importance_tables = {}

# ------------------------------------------------------------
# 1. LINEAR REGRESSION
# ------------------------------------------------------------

print("\nCalculating SHAP — Linear Regression...")

lr_explainer = shap.LinearExplainer(
    lr_model,
    X_train
)

lr_shap = lr_explainer(X_shap)

lr_importance = np.abs(lr_shap.values).mean(axis=0)

lr_table = pd.DataFrame({
    "Feature": final_features,
    "Mean_Abs_SHAP": lr_importance
}).sort_values(
    "Mean_Abs_SHAP",
    ascending=False
).reset_index(drop=True)

lr_table["Rank"] = np.arange(1, len(lr_table) + 1)

shap_importance_tables["Linear Regression"] = lr_table
shap_rankings["Linear Regression"] = lr_table["Feature"].tolist()


# ------------------------------------------------------------
# 2. RANDOM FOREST
# ------------------------------------------------------------

print("Calculating SHAP — Random Forest...")

rf_explainer = shap.TreeExplainer(rf_model)
rf_shap = rf_explainer(X_shap)

rf_importance = np.abs(rf_shap.values).mean(axis=0)

rf_table = pd.DataFrame({
    "Feature": final_features,
    "Mean_Abs_SHAP": rf_importance
}).sort_values(
    "Mean_Abs_SHAP",
    ascending=False
).reset_index(drop=True)

rf_table["Rank"] = np.arange(1, len(rf_table) + 1)

shap_importance_tables["Random Forest"] = rf_table
shap_rankings["Random Forest"] = rf_table["Feature"].tolist()


# ------------------------------------------------------------
# 3. XGBOOST
# ------------------------------------------------------------

print("Calculating SHAP — XGBoost...")

xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap = xgb_explainer(X_shap)

xgb_importance = np.abs(xgb_shap.values).mean(axis=0)

xgb_table = pd.DataFrame({
    "Feature": final_features,
    "Mean_Abs_SHAP": xgb_importance
}).sort_values(
    "Mean_Abs_SHAP",
    ascending=False
).reset_index(drop=True)

xgb_table["Rank"] = np.arange(1, len(xgb_table) + 1)

shap_importance_tables["XGBoost"] = xgb_table
shap_rankings["XGBoost"] = xgb_table["Feature"].tolist()


# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------

for model_name, table in shap_importance_tables.items():

    print("\n" + "=" * 65)
    print(model_name.upper())
    print("=" * 65)

    print(
        table.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


# ------------------------------------------------------------
# TOP-5 COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TOP-5 SHAP FEATURES")
print("=" * 65)

for model_name, ranking in shap_rankings.items():
    print(f"{model_name:20s}: {ranking[:5]}")

In [ ]:
# ============================================================
# CROSS-MODEL SHAP RANKING AGREEMENT
# Revised 16-feature analysis
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from itertools import combinations

model_names = list(shap_rankings.keys())

# ------------------------------------------------------------
# Convert feature orderings into rank dictionaries
# ------------------------------------------------------------

rank_dicts = {}

for model_name, ranking in shap_rankings.items():
    rank_dicts[model_name] = {
        feature: rank
        for rank, feature in enumerate(ranking, start=1)
    }

# ------------------------------------------------------------
# 1. Spearman rank correlation
# ------------------------------------------------------------

spearman_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)

spearman_values = []

for m1 in model_names:
    for m2 in model_names:

        ranks1 = [rank_dicts[m1][f] for f in final_features]
        ranks2 = [rank_dicts[m2][f] for f in final_features]

        rho, _ = spearmanr(ranks1, ranks2)

        spearman_matrix.loc[m1, m2] = rho

for m1, m2 in combinations(model_names, 2):

    ranks1 = [rank_dicts[m1][f] for f in final_features]
    ranks2 = [rank_dicts[m2][f] for f in final_features]

    rho, _ = spearmanr(ranks1, ranks2)

    spearman_values.append(rho)

# ------------------------------------------------------------
# 2. Top-5 Jaccard similarity
# ------------------------------------------------------------

jaccard_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)

jaccard_values = []

for m1 in model_names:
    for m2 in model_names:

        top5_1 = set(shap_rankings[m1][:5])
        top5_2 = set(shap_rankings[m2][:5])

        jaccard = len(top5_1 & top5_2) / len(top5_1 | top5_2)

        jaccard_matrix.loc[m1, m2] = jaccard

for m1, m2 in combinations(model_names, 2):

    top5_1 = set(shap_rankings[m1][:5])
    top5_2 = set(shap_rankings[m2][:5])

    jaccard = len(top5_1 & top5_2) / len(top5_1 | top5_2)

    jaccard_values.append(jaccard)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 70)
print("SPEARMAN RANK CORRELATION — 16 FEATURES")
print("=" * 70)

print(spearman_matrix.round(4))

print("\nPairwise Spearman values:")
for (m1, m2), value in zip(
    combinations(model_names, 2),
    spearman_values
):
    print(f"{m1} vs {m2}: {value:.4f}")

print(
    f"\nMean pairwise Spearman: "
    f"{np.mean(spearman_values):.4f} ± "
    f"{np.std(spearman_values, ddof=0):.4f}"
)

print("\n" + "=" * 70)
print("TOP-5 JACCARD SIMILARITY — 16 FEATURES")
print("=" * 70)

print(jaccard_matrix.round(4))

print("\nPairwise Top-5 Jaccard:")
for (m1, m2), value in zip(
    combinations(model_names, 2),
    jaccard_values
):
    print(f"{m1} vs {m2}: {value:.4f}")

print(
    f"\nMean pairwise Top-5 Jaccard: "
    f"{np.mean(jaccard_values):.4f} ± "
    f"{np.std(jaccard_values, ddof=0):.4f}"
)

In [ ]:
# ============================================================
# RETRAIN SVR — FINAL 16-FEATURE SET
# Exact configuration from original experiment
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import time

svr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",
        C=100,
        epsilon=0.1,
        gamma="scale"
    ))
])

print("Training SVR...")
start = time.time()

svr_model.fit(X_train, y_train)

svr_predictions = svr_model.predict(X_test)

svr_rmse = np.sqrt(mean_squared_error(y_test, svr_predictions))
svr_mae = mean_absolute_error(y_test, svr_predictions)
svr_r2 = r2_score(y_test, svr_predictions)

elapsed = time.time() - start

print("\n" + "=" * 60)
print("SVR — REVISED 16-FEATURE MODEL")
print("=" * 60)
print(f"RMSE : {svr_rmse:.4f}")
print(f"MAE  : {svr_mae:.4f}")
print(f"R²   : {svr_r2:.4f}")
print(f"Time : {elapsed:.2f} seconds")

print("\nConfiguration:")
print("Scaler  : StandardScaler")
print("Kernel  : RBF")
print("C       : 100")
print("epsilon : 0.1")
print("gamma   : scale")

In [ ]:
# ============================================================
# SVR KERNELSHAP — REVISED 16-FEATURE MODEL
# Same sampling strategy as original SVR SHAP experiment
# ============================================================

import shap
import numpy as np
import pandas as pd
import time

# Fixed samples for reproducibility
X_background = shap.sample(
    X_train,
    50,
    random_state=42
)

X_svr_shap = X_test.sample(
    n=100,
    random_state=42
)

# Prediction wrapper preserves the full scaler + SVR pipeline
def svr_predict(X):
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=final_features)
    return svr_model.predict(X)

print("Calculating SVR KernelSHAP...")
print(f"Background samples : {len(X_background)}")
print(f"Explanation samples: {len(X_svr_shap)}")

start = time.time()

svr_explainer = shap.KernelExplainer(
    svr_predict,
    X_background
)

svr_shap_values = svr_explainer.shap_values(
    X_svr_shap
)

elapsed = time.time() - start

# Mean absolute SHAP importance
svr_importance = np.abs(svr_shap_values).mean(axis=0)

svr_table = pd.DataFrame({
    "Feature": final_features,
    "Mean_Abs_SHAP": svr_importance
}).sort_values(
    "Mean_Abs_SHAP",
    ascending=False
).reset_index(drop=True)

svr_table["Rank"] = np.arange(1, len(svr_table) + 1)

# Add to existing dictionaries
shap_importance_tables["SVR"] = svr_table
shap_rankings["SVR"] = svr_table["Feature"].tolist()

print("\n" + "=" * 65)
print("SVR — KERNELSHAP — 16 FEATURES")
print("=" * 65)

print(
    svr_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nTop-5 SVR features:")
print(shap_rankings["SVR"][:5])

print(f"\nKernelSHAP computation time: {elapsed/60:.2f} minutes")

In [ ]:
# ============================================================
# FINAL FOUR-MODEL SHAP AGREEMENT + RANDOM BASELINE
# Revised 16-feature analysis
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from itertools import combinations

model_names = [
    "Linear Regression",
    "Random Forest",
    "XGBoost",
    "SVR"
]

# ------------------------------------------------------------
# 1. Verify rankings are available
# ------------------------------------------------------------

print("=" * 75)
print("FINAL SHAP RANKINGS — 16 RETAINED FEATURES")
print("=" * 75)

for model in model_names:
    print(f"\n{model}:")
    print(shap_rankings[model])

# ------------------------------------------------------------
# 2. Convert feature orderings to numerical ranks
# ------------------------------------------------------------

rank_dicts = {}

for model, ranking in shap_rankings.items():
    rank_dicts[model] = {
        feature: rank
        for rank, feature in enumerate(ranking, start=1)
    }

# ------------------------------------------------------------
# 3. Spearman rank correlation
# ------------------------------------------------------------

spearman_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)

spearman_values = []

for m1 in model_names:
    for m2 in model_names:

        ranks1 = [rank_dicts[m1][f] for f in final_features]
        ranks2 = [rank_dicts[m2][f] for f in final_features]

        rho, _ = spearmanr(ranks1, ranks2)

        spearman_matrix.loc[m1, m2] = rho

for m1, m2 in combinations(model_names, 2):

    ranks1 = [rank_dicts[m1][f] for f in final_features]
    ranks2 = [rank_dicts[m2][f] for f in final_features]

    rho, _ = spearmanr(ranks1, ranks2)
    spearman_values.append(rho)

# ------------------------------------------------------------
# 4. Top-5 Jaccard
# ------------------------------------------------------------

jaccard_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)

jaccard_values = []

for m1 in model_names:
    for m2 in model_names:

        a = set(shap_rankings[m1][:5])
        b = set(shap_rankings[m2][:5])

        j = len(a & b) / len(a | b)

        jaccard_matrix.loc[m1, m2] = j

for m1, m2 in combinations(model_names, 2):

    a = set(shap_rankings[m1][:5])
    b = set(shap_rankings[m2][:5])

    jaccard_values.append(
        len(a & b) / len(a | b)
    )

# ------------------------------------------------------------
# 5. Random-ranking baseline
# ------------------------------------------------------------

N_SIMULATIONS = 10000
N_FEATURES = len(final_features)
TOP_K = 5

rng = np.random.default_rng(42)

random_spearman = np.empty(N_SIMULATIONS)
random_jaccard = np.empty(N_SIMULATIONS)

for i in range(N_SIMULATIONS):

    r1 = rng.permutation(N_FEATURES)
    r2 = rng.permutation(N_FEATURES)

    random_spearman[i] = spearmanr(r1, r2)[0]

    top1 = set(r1[:TOP_K])
    top2 = set(r2[:TOP_K])

    random_jaccard[i] = (
        len(top1 & top2) /
        len(top1 | top2)
    )

# ------------------------------------------------------------
# 6. Print observed results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FOUR-MODEL SPEARMAN CORRELATION")
print("=" * 75)

print(spearman_matrix.round(4))

print("\nPairwise values:")

for (m1, m2), value in zip(
    combinations(model_names, 2),
    spearman_values
):
    print(f"{m1} vs {m2}: {value:.4f}")

print(
    f"\nMean pairwise Spearman: "
    f"{np.mean(spearman_values):.4f} ± "
    f"{np.std(spearman_values, ddof=0):.4f}"
)

print("\n" + "=" * 75)
print("FOUR-MODEL TOP-5 JACCARD")
print("=" * 75)

print(jaccard_matrix.round(4))

print("\nPairwise values:")

for (m1, m2), value in zip(
    combinations(model_names, 2),
    jaccard_values
):
    print(f"{m1} vs {m2}: {value:.4f}")

print(
    f"\nMean pairwise Top-5 Jaccard: "
    f"{np.mean(jaccard_values):.4f} ± "
    f"{np.std(jaccard_values, ddof=0):.4f}"
)

# ------------------------------------------------------------
# 7. Print random baseline
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("RANDOM-RANKING BASELINE")
print("=" * 75)

print(f"Features    : {N_FEATURES}")
print(f"Top-k       : {TOP_K}")
print(f"Simulations : {N_SIMULATIONS}")
print("Random seed : 42")

print("\nRandom Spearman:")
print(f"Mean  : {random_spearman.mean():.4f}")
print(f"SD    : {random_spearman.std(ddof=0):.4f}")
print(
    f"95% range: "
    f"{np.percentile(random_spearman, 2.5):.4f} to "
    f"{np.percentile(random_spearman, 97.5):.4f}"
)

print("\nRandom Top-5 Jaccard:")
print(f"Mean  : {random_jaccard.mean():.4f}")
print(f"SD    : {random_jaccard.std(ddof=0):.4f}")
print(
    f"95% range: "
    f"{np.percentile(random_jaccard, 2.5):.4f} to "
    f"{np.percentile(random_jaccard, 97.5):.4f}"
)

# ------------------------------------------------------------
# 8. Empirical exceedance rates
# ------------------------------------------------------------

observed_spearman = np.mean(spearman_values)
observed_jaccard = np.mean(jaccard_values)

spearman_exceedance = np.mean(
    random_spearman >= observed_spearman
)

jaccard_exceedance = np.mean(
    random_jaccard >= observed_jaccard
)

print("\nObserved vs random:")
print(f"Observed mean Spearman : {observed_spearman:.4f}")
print(f"Observed mean Jaccard  : {observed_jaccard:.4f}")

print(
    f"Random Spearman >= observed: "
    f"{spearman_exceedance:.6f}"
)

print(
    f"Random Jaccard >= observed : "
    f"{jaccard_exceedance:.6f}"
)

In [ ]:
# ============================================================
# FEATURE–RUL CORRELATION vs SHAP IMPORTANCE
# Training data only — 16 retained features
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

# ------------------------------------------------------------
# 1. Calculate feature–RUL correlations on TRAINING DATA ONLY
# ------------------------------------------------------------

correlation_results = []

for feature in final_features:

    x = X_train[feature].values
    y = y_train.values

    pearson_r, _ = pearsonr(x, y)
    spearman_r, _ = spearmanr(x, y)

    correlation_results.append({
        "Feature": feature,
        "Pearson_r": pearson_r,
        "Abs_Pearson": abs(pearson_r),
        "Spearman_r": spearman_r,
        "Abs_Spearman": abs(spearman_r)
    })

correlation_df = pd.DataFrame(correlation_results)

# Rank by absolute correlation
correlation_df["Pearson_Rank"] = (
    correlation_df["Abs_Pearson"]
    .rank(ascending=False, method="min")
    .astype(int)
)

correlation_df["Spearman_Rank"] = (
    correlation_df["Abs_Spearman"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# Sort for display by absolute Spearman correlation
correlation_df = correlation_df.sort_values(
    "Abs_Spearman",
    ascending=False
).reset_index(drop=True)

print("=" * 85)
print("FEATURE–RUL CORRELATION — TRAINING DATA ONLY")
print("=" * 85)

print(
    correlation_df[
        [
            "Feature",
            "Pearson_r",
            "Abs_Pearson",
            "Pearson_Rank",
            "Spearman_r",
            "Abs_Spearman",
            "Spearman_Rank"
        ]
    ].round(4).to_string(index=False)
)


# ------------------------------------------------------------
# 2. Create correlation rankings
# ------------------------------------------------------------

pearson_ranking = (
    correlation_df
    .sort_values("Abs_Pearson", ascending=False)["Feature"]
    .tolist()
)

spearman_ranking = (
    correlation_df
    .sort_values("Abs_Spearman", ascending=False)["Feature"]
    .tolist()
)

print("\n" + "=" * 85)
print("CORRELATION-BASED FEATURE RANKINGS")
print("=" * 85)

print("\nPearson |r| ranking:")
print(pearson_ranking)

print("\nSpearman |rho| ranking:")
print(spearman_ranking)


# ------------------------------------------------------------
# 3. Function: convert ordered feature list to numerical ranks
# ------------------------------------------------------------

def ranking_to_dict(ranking):
    return {
        feature: rank
        for rank, feature in enumerate(ranking, start=1)
    }


pearson_rank_dict = ranking_to_dict(pearson_ranking)
spearman_rank_dict = ranking_to_dict(spearman_ranking)


# ------------------------------------------------------------
# 4. Compare correlation rankings with SHAP rankings
# ------------------------------------------------------------

comparison_results = []

for model in model_names:

    shap_ranking = shap_rankings[model]
    shap_rank_dict = ranking_to_dict(shap_ranking)

    # Full-ranking agreement:
    # Spearman correlation between numerical feature ranks
    shap_ranks = [
        shap_rank_dict[f] for f in final_features
    ]

    pearson_ranks = [
        pearson_rank_dict[f] for f in final_features
    ]

    spearman_ranks = [
        spearman_rank_dict[f] for f in final_features
    ]

    rho_shap_pearson, _ = spearmanr(
        shap_ranks,
        pearson_ranks
    )

    rho_shap_spearman, _ = spearmanr(
        shap_ranks,
        spearman_ranks
    )

    # Top-5 Jaccard
    shap_top5 = set(shap_ranking[:5])
    pearson_top5 = set(pearson_ranking[:5])
    spearman_top5 = set(spearman_ranking[:5])

    jaccard_pearson = (
        len(shap_top5 & pearson_top5) /
        len(shap_top5 | pearson_top5)
    )

    jaccard_spearman = (
        len(shap_top5 & spearman_top5) /
        len(shap_top5 | spearman_top5)
    )

    comparison_results.append({
        "Model": model,
        "SHAP_vs_Pearson_Spearman": rho_shap_pearson,
        "SHAP_vs_Spearman_Spearman": rho_shap_spearman,
        "Top5_Jaccard_Pearson": jaccard_pearson,
        "Top5_Jaccard_Spearman": jaccard_spearman
    })


comparison_df = pd.DataFrame(comparison_results)

print("\n" + "=" * 85)
print("SHAP vs SIMPLE FEATURE–RUL CORRELATION")
print("=" * 85)

print(
    comparison_df
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Display Top-5 sets clearly
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("TOP-5 FEATURE COMPARISON")
print("=" * 85)

print("\nPearson Top-5:")
print(pearson_ranking[:5])

print("\nSpearman Top-5:")
print(spearman_ranking[:5])

for model in model_names:
    print(f"\n{model} SHAP Top-5:")
    print(shap_rankings[model][:5])


# ------------------------------------------------------------
# 6. Mean agreement across four models
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("MEAN SHAP–CORRELATION AGREEMENT")
print("=" * 85)

print(
    "Mean SHAP vs Pearson ranking Spearman: "
    f"{comparison_df['SHAP_vs_Pearson_Spearman'].mean():.4f}"
)

print(
    "Mean SHAP vs Spearman-correlation ranking Spearman: "
    f"{comparison_df['SHAP_vs_Spearman_Spearman'].mean():.4f}"
)

print(
    "Mean Top-5 Jaccard vs Pearson: "
    f"{comparison_df['Top5_Jaccard_Pearson'].mean():.4f}"
)

print(
    "Mean Top-5 Jaccard vs Spearman correlation: "
    f"{comparison_df['Top5_Jaccard_Spearman'].mean():.4f}"
)

In [ ]:
print("Available possible dataframes:")

for name in [
    "df",
    "data",
    "df_raw",
    "train_data",
    "test_data",
    "X",
    "X_train",
    "X_test"
]:
    if name in globals():
        obj = globals()[name]
        try:
            print(
                f"{name:12s} shape={obj.shape} "
                f"sensor_6={'sensor_6' in obj.columns}"
            )
        except:
            print(name, "exists")

In [ ]:
# ============================================================
# SENSOR_6 SENSITIVITY ANALYSIS
# 16 retained features vs 17 features (sensor_6 restored)
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor


# ------------------------------------------------------------
# 1. Restore sensor_6
# ------------------------------------------------------------

features_with_s6 = final_features + ["sensor_6"]

X_train_s6 = train_data[features_with_s6].copy()
X_test_s6  = test_data[features_with_s6].copy()

print("=" * 80)
print("SENSOR_6 SENSITIVITY ANALYSIS")
print("=" * 80)

print(f"Current retained features : {len(final_features)}")
print(f"Features with sensor_6    : {len(features_with_s6)}")
print(f"Training rows             : {len(X_train_s6)}")
print(f"Test rows                 : {len(X_test_s6)}")


# ------------------------------------------------------------
# 2. Exact same model configurations
# ------------------------------------------------------------

models_s6 = {

    "Linear Regression":
        LinearRegression(),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=150,
            max_depth=15,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        ),

    "XGBoost":
        XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            objective="reg:squarederror"
        ),

    "SVR":
        Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(
                kernel="rbf",
                C=100,
                epsilon=0.1,
                gamma="scale"
            ))
        ])
}


# ------------------------------------------------------------
# 3. Train models with sensor_6 restored
# ------------------------------------------------------------

results_s6 = []

for name, model in models_s6.items():

    print(f"\nTraining {name}...")

    start = time.time()

    model.fit(X_train_s6, y_train)

    pred = model.predict(X_test_s6)

    elapsed = time.time() - start

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    results_s6.append({
        "Model": name,
        "RMSE_with_s6": rmse,
        "MAE_with_s6": mae,
        "R2_with_s6": r2,
        "Time_sec": elapsed
    })

    print(
        f"RMSE={rmse:.4f} | "
        f"MAE={mae:.4f} | "
        f"R²={r2:.4f} | "
        f"Time={elapsed:.2f}s"
    )


results_s6_df = pd.DataFrame(results_s6)


# ------------------------------------------------------------
# 4. Current authoritative 16-feature results
# ------------------------------------------------------------

baseline_16 = pd.DataFrame({

    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "SVR"
    ],

    "RMSE_without_s6": [
        20.4330,
        17.0451,
        17.0251,
        17.6777
    ],

    "MAE_without_s6": [
        16.5135,
        12.3499,
        12.2121,
        11.4549
    ],

    "R2_without_s6": [
        0.7601,
        0.8330,
        0.8334,
        0.8204
    ]
})


# ------------------------------------------------------------
# 5. Compare results
# ------------------------------------------------------------

comparison_s6 = baseline_16.merge(
    results_s6_df,
    on="Model"
)

comparison_s6["RMSE_change"] = (
    comparison_s6["RMSE_with_s6"] -
    comparison_s6["RMSE_without_s6"]
)

comparison_s6["RMSE_change_percent"] = (
    comparison_s6["RMSE_change"] /
    comparison_s6["RMSE_without_s6"]
) * 100

comparison_s6["MAE_change"] = (
    comparison_s6["MAE_with_s6"] -
    comparison_s6["MAE_without_s6"]
)

comparison_s6["R2_change"] = (
    comparison_s6["R2_with_s6"] -
    comparison_s6["R2_without_s6"]
)


print("\n" + "=" * 100)
print("FINAL SENSOR_6 SENSITIVITY COMPARISON")
print("=" * 100)

columns_to_show = [
    "Model",
    "RMSE_without_s6",
    "RMSE_with_s6",
    "RMSE_change",
    "RMSE_change_percent",
    "MAE_without_s6",
    "MAE_with_s6",
    "R2_without_s6",
    "R2_with_s6"
]

print(
    comparison_s6[columns_to_show]
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. sensor_6 characteristics
# ------------------------------------------------------------

dominant_fraction = (
    train_data["sensor_6"]
    .value_counts(normalize=True)
    .iloc[0]
)

print("\n" + "=" * 80)
print("SENSOR_6 CHARACTERISTICS")
print("=" * 80)

print("Unique values:",
      train_data["sensor_6"].nunique())

print("Dominant value:",
      train_data["sensor_6"].mode().iloc[0])

print(
    f"Dominant-value fraction: "
    f"{dominant_fraction:.6f} "
    f"({dominant_fraction * 100:.4f}%)"
)

print(
    f"Variance: "
    f"{train_data['sensor_6'].var():.10f}"
)

In [ ]:
import os

print("Searching for FD001 files...\n")

found_files = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "FD001" in file:
            path = os.path.join(root, file)
            found_files.append(path)
            print(path)

print(f"\nTotal FD001 files found: {len(found_files)}")

In [ ]:
import pandas as pd

# ---------------------------------------------------------
# Official NASA C-MAPSS FD001 test data
# ---------------------------------------------------------

test_path = TEST_PATH
rul_path  = RUL_PATH

# Same column structure used for the training data
columns = (
    ["engine_id", "cycle",
     "op_setting_1", "op_setting_2", "op_setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

# Load official test trajectories
official_test = pd.read_csv(
    test_path,
    sep=r"\s+",
    header=None,
    names=columns
)

# Load NASA-provided RUL values
official_rul = pd.read_csv(
    rul_path,
    sep=r"\s+",
    header=None,
    names=["final_RUL"]
)

print("Official test shape:", official_test.shape)
print("Number of test engines:", official_test["engine_id"].nunique())
print("RUL file shape:", official_rul.shape)

print("\nFirst 5 rows of official test data:")
display(official_test.head())

print("\nFirst 10 NASA RUL values:")
display(official_rul.head(10))

print("\nLast observed cycle for first 10 engines:")
display(
    official_test.groupby("engine_id")["cycle"]
    .max()
    .head(10)
)

In [ ]:
# ---------------------------------------------------------
# Construct RUL targets for official FD001 test trajectories
# ---------------------------------------------------------

# NASA RUL file corresponds to engine IDs 1, 2, ..., 100
official_rul = official_rul.copy()
official_rul["engine_id"] = range(1, len(official_rul) + 1)

# Last observed cycle of each test engine
last_cycles = (
    official_test.groupby("engine_id")["cycle"]
    .max()
    .reset_index(name="last_observed_cycle")
)

# Attach NASA-provided RUL at the last observed cycle
engine_rul_info = last_cycles.merge(
    official_rul,
    on="engine_id",
    how="left"
)

# Implied failure cycle
engine_rul_info["failure_cycle"] = (
    engine_rul_info["last_observed_cycle"]
    + engine_rul_info["final_RUL"]
)

# Attach failure cycle to every observation
official_test_eval = official_test.merge(
    engine_rul_info[
        ["engine_id", "last_observed_cycle",
         "final_RUL", "failure_cycle"]
    ],
    on="engine_id",
    how="left"
)

# Uncapped RUL for every observed cycle
official_test_eval["RUL_uncapped"] = (
    official_test_eval["failure_cycle"]
    - official_test_eval["cycle"]
)

# Apply the same cap used during model training
RUL_CAP = 125
official_test_eval["RUL"] = (
    official_test_eval["RUL_uncapped"]
    .clip(upper=RUL_CAP)
)

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("Evaluation dataset shape:", official_test_eval.shape)
print("Number of engines:", official_test_eval["engine_id"].nunique())
print("Missing RUL values:", official_test_eval["RUL"].isna().sum())

print("\nEngine-level information — first 10:")
display(engine_rul_info.head(10))

print("\nEngine 1 — last 5 observed cycles:")
display(
    official_test_eval.loc[
        official_test_eval["engine_id"] == 1,
        ["engine_id", "cycle",
         "RUL_uncapped", "RUL"]
    ].tail()
)

print("\nRUL summary after 125-cycle cap:")
display(official_test_eval["RUL"].describe())

# Critical consistency check:
last_rows = (
    official_test_eval
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
)

check = (
    last_rows["RUL_uncapped"].to_numpy()
    == official_rul.sort_values("engine_id")["final_RUL"].to_numpy()
)

print("\nNASA final-RUL consistency check:", check.all())

In [ ]:
# ---------------------------------------------------------
# Prepare official FD001 benchmark set
# One final observation per test engine
# ---------------------------------------------------------

official_final = (
    official_test_eval
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

# Use exactly the same 16 retained predictors
X_official = official_final[final_features]
y_official = official_final["RUL"]

print("Official benchmark observations:", len(official_final))
print("Number of engines:", official_final["engine_id"].nunique())
print("Number of predictors:", X_official.shape[1])

print("\nFeatures used:")
print(list(X_official.columns))

print("\nOfficial capped RUL summary:")
display(y_official.describe())

print("\nFirst 10 benchmark engines:")
display(
    official_final[
        ["engine_id", "cycle", "final_RUL",
         "RUL_uncapped", "RUL"]
    ].head(10)
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------------------------------------------------------
# Official FD001 benchmark evaluation
# ---------------------------------------------------------

models_official = {
    "Linear Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "SVR": svr_model
}

official_results = []
official_predictions = {}

for name, model in models_official.items():

    # Predict official final-cycle observations
    y_pred = model.predict(X_official)

    official_predictions[name] = y_pred

    rmse = np.sqrt(mean_squared_error(y_official, y_pred))
    mae = mean_absolute_error(y_official, y_pred)
    r2 = r2_score(y_official, y_pred)

    official_results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

official_results_df = pd.DataFrame(official_results)

print("OFFICIAL NASA C-MAPSS FD001 TEST-SET RESULTS")
print("=" * 60)

display(
    official_results_df.style.format({
        "RMSE": "{:.4f}",
        "MAE": "{:.4f}",
        "R2": "{:.4f}"
    })
)

# ---------------------------------------------------------
# Compare with our internal 80/20 engine-level test results
# ---------------------------------------------------------

internal_results = {
    "Linear Regression": 20.4330,
    "Random Forest": 17.0451,
    "XGBoost": 17.0251,
    "SVR": 17.6777
}

comparison = official_results_df[["Model", "RMSE"]].copy()
comparison["Internal_80_20_RMSE"] = comparison["Model"].map(internal_results)
comparison["Official_FD001_RMSE"] = comparison["RMSE"]
comparison["Difference"] = (
    comparison["Official_FD001_RMSE"]
    - comparison["Internal_80_20_RMSE"]
)

comparison = comparison[
    [
        "Model",
        "Internal_80_20_RMSE",
        "Official_FD001_RMSE",
        "Difference"
    ]
]

print("\nINTERNAL SPLIT vs OFFICIAL NASA TEST SET")
print("=" * 60)

display(
    comparison.style.format({
        "Internal_80_20_RMSE": "{:.4f}",
        "Official_FD001_RMSE": "{:.4f}",
        "Difference": "{:+.4f}"
    })
)

In [ ]:
names_to_check = [
    "X_train",
    "X_test",
    "y_train",
    "y_test",
    "xgb_model",
    "train_data",
    "test_data"
]

for name in names_to_check:
    print(name, ":", name in globals())

In [ ]:
print("train_data shape:", train_data.shape)
print("test_data shape :", test_data.shape)

print("\nTraining engines:",
      train_data["engine_id"].nunique())

print("Test engines:",
      test_data["engine_id"].nunique())

print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTrain engine IDs:")
print(sorted(train_data["engine_id"].unique()))

print("\nTest engine IDs:")
print(sorted(test_data["engine_id"].unique()))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ============================================================
# SPLIT EXISTING TRAINING ENGINES:
# proper training vs conformal calibration
# ============================================================

train_engine_ids = np.array(sorted(train_data["engine_id"].unique()))

proper_engine_ids, calib_engine_ids = train_test_split(
    train_engine_ids,
    test_size=0.20,
    random_state=42
)

# Row masks
proper_mask = train_data["engine_id"].isin(proper_engine_ids).to_numpy()
calib_mask  = train_data["engine_id"].isin(calib_engine_ids).to_numpy()

# Same 16 retained features
X_proper = train_data.loc[proper_mask, final_features]
y_proper = train_data.loc[proper_mask, "RUL"]

X_calib = train_data.loc[calib_mask, final_features]
y_calib = train_data.loc[calib_mask, "RUL"]

print("CONFORMAL DATA SPLIT")
print("=" * 55)

print("Proper-training engines :", len(proper_engine_ids))
print("Calibration engines     :", len(calib_engine_ids))
print("Final test engines      :", test_data['engine_id'].nunique())

print("\nProper-training rows :", len(X_proper))
print("Calibration rows     :", len(X_calib))
print("Final test rows      :", len(X_test))

print("\nEngine overlap checks:")
print("Proper vs calibration:",
      len(set(proper_engine_ids) & set(calib_engine_ids)))

print("Proper vs final test:",
      len(set(proper_engine_ids) &
          set(test_data["engine_id"].unique())))

print("Calibration vs final test:",
      len(set(calib_engine_ids) &
          set(test_data["engine_id"].unique())))

print("\nProper-training engines:")
print(sorted(proper_engine_ids))

print("\nCalibration engines:")
print(sorted(calib_engine_ids))

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ============================================================
# SPLIT-CONFORMAL PREDICTION INTERVAL — XGBOOST
# ============================================================

# Same frozen XGBoost configuration
xgb_conformal = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

# 1. Train ONLY on proper-training engines
xgb_conformal.fit(X_proper, y_proper)

# 2. Calibration predictions and absolute residuals
calib_pred = xgb_conformal.predict(X_calib)
calib_errors = np.abs(y_calib.to_numpy() - calib_pred)

# 3. Finite-sample conformal 95% quantile
alpha = 0.05
n_cal = len(calib_errors)

q_level = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal
q_level = min(q_level, 1.0)

q_hat = np.quantile(
    calib_errors,
    q_level,
    method="higher"
)

# 4. Predictions on untouched final test set
test_pred_conf = xgb_conformal.predict(X_test)

# Raw symmetric conformal bounds
lower_raw = test_pred_conf - q_hat
upper_raw = test_pred_conf + q_hat

# Operationally bounded RUL interval
# RUL target itself is constrained to [0, 125]
lower_bounded = np.maximum(0, lower_raw)
upper_bounded = np.minimum(125, upper_raw)

y_true = y_test.to_numpy()

# ------------------------------------------------------------
# Coverage
# ------------------------------------------------------------

covered_raw = (
    (y_true >= lower_raw) &
    (y_true <= upper_raw)
)

covered_bounded = (
    (y_true >= lower_bounded) &
    (y_true <= upper_bounded)
)

overall_raw_coverage = covered_raw.mean()
overall_bounded_coverage = covered_bounded.mean()

# Near failure: same thesis definition RUL < 30
near_failure = y_true < 30

near_failure_raw_coverage = covered_raw[near_failure].mean()
near_failure_bounded_coverage = covered_bounded[near_failure].mean()

# Other RUL regions for diagnostic purposes
mid_life = (y_true >= 30) & (y_true < 125)
capped_region = y_true == 125

# ------------------------------------------------------------
# Interval widths
# ------------------------------------------------------------

raw_width = upper_raw - lower_raw
bounded_width = upper_bounded - lower_bounded

# ------------------------------------------------------------
# Point prediction performance of conformal model
# ------------------------------------------------------------

rmse_conf = np.sqrt(mean_squared_error(y_true, test_pred_conf))
mae_conf = mean_absolute_error(y_true, test_pred_conf)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("95% SPLIT-CONFORMAL VALIDATION — XGBOOST")
print("=" * 65)

print(f"Calibration observations       : {n_cal}")
print(f"Conformal quantile level       : {q_level:.6f}")
print(f"Conformal q_hat                : {q_hat:.4f} cycles")

print(f"\nPoint-prediction RMSE          : {rmse_conf:.4f}")
print(f"Point-prediction MAE           : {mae_conf:.4f}")

print("\nCOVERAGE")
print("-" * 65)
print(f"Nominal coverage               : 95.00%")
print(f"Overall raw coverage           : {overall_raw_coverage*100:.2f}%")
print(f"Overall bounded coverage       : {overall_bounded_coverage*100:.2f}%")

print(f"\nNear-failure observations RUL<30: {near_failure.sum()}")
print(f"Near-failure raw coverage        : {near_failure_raw_coverage*100:.2f}%")
print(f"Near-failure bounded coverage    : {near_failure_bounded_coverage*100:.2f}%")

if mid_life.sum() > 0:
    print(f"\nRUL 30–124 observations         : {mid_life.sum()}")
    print(f"RUL 30–124 raw coverage         : {covered_raw[mid_life].mean()*100:.2f}%")
    print(f"RUL 30–124 bounded coverage     : {covered_bounded[mid_life].mean()*100:.2f}%")

if capped_region.sum() > 0:
    print(f"\nRUL=125 observations            : {capped_region.sum()}")
    print(f"RUL=125 raw coverage            : {covered_raw[capped_region].mean()*100:.2f}%")
    print(f"RUL=125 bounded coverage        : {covered_bounded[capped_region].mean()*100:.2f}%")

print("\nINTERVAL WIDTH")
print("-" * 65)
print(f"Raw interval width             : {raw_width.mean():.4f} cycles")
print(f"Expected raw width (2*q_hat)   : {2*q_hat:.4f} cycles")
print(f"Bounded mean interval width    : {bounded_width.mean():.4f} cycles")
print(f"Bounded minimum width          : {bounded_width.min():.4f} cycles")
print(f"Bounded maximum width          : {bounded_width.max():.4f} cycles")

In [ ]:
# ============================================================
# NOISE ROBUSTNESS — STEP 1
# Training-feature scale verification
# ============================================================

feature_noise_stats = pd.DataFrame({
    "Feature": final_features,
    "Train_Mean": X_train[final_features].mean().values,
    "Train_SD": X_train[final_features].std(ddof=0).values
})

print("FEATURE SCALES USED FOR NOISE INJECTION")
print("=" * 65)

display(
    feature_noise_stats.style.format({
        "Train_Mean": "{:.6f}",
        "Train_SD": "{:.6f}"
    })
)

print("\nNumber of features:", len(final_features))
print(
    "Features with zero training SD:",
    (feature_noise_stats["Train_SD"] == 0).sum()
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# ============================================================
# NOISE ROBUSTNESS — ALL FOUR MODELS
# ============================================================

noise_levels = [0.00, 0.01, 0.05, 0.10, 0.20]

models_noise = {
    "Linear Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "SVR": svr_model
}

# Training SD determines noise scale
train_sd = X_train[final_features].std(ddof=0)

# Fixed seed for reproducibility
rng = np.random.default_rng(42)

results = []

for noise_level in noise_levels:

    # Start from the original untouched test set
    X_noisy = X_test[final_features].copy()

    if noise_level > 0:

        # Generate independent Gaussian noise
        noise = rng.normal(
            loc=0.0,
            scale=1.0,
            size=X_noisy.shape
        )

        # Scale noise feature-by-feature
        noise = noise * train_sd.to_numpy() * noise_level

        X_noisy = X_noisy + noise

    for model_name, model in models_noise.items():

        y_pred = model.predict(X_noisy)

        rmse = np.sqrt(
            mean_squared_error(y_test, y_pred)
        )

        results.append({
            "Model": model_name,
            "Noise_Level": noise_level,
            "Noise_Percent": noise_level * 100,
            "RMSE": rmse
        })

noise_results_all = pd.DataFrame(results)

# Baseline RMSE for each model
baseline_rmse = (
    noise_results_all[
        noise_results_all["Noise_Level"] == 0
    ]
    .set_index("Model")["RMSE"]
)

noise_results_all["Baseline_RMSE"] = (
    noise_results_all["Model"].map(baseline_rmse)
)

noise_results_all["RMSE_Change"] = (
    noise_results_all["RMSE"]
    - noise_results_all["Baseline_RMSE"]
)

noise_results_all["RMSE_Change_Percent"] = (
    noise_results_all["RMSE_Change"]
    / noise_results_all["Baseline_RMSE"]
    * 100
)

# ------------------------------------------------------------
# Display RMSE table
# ------------------------------------------------------------

rmse_table = noise_results_all.pivot(
    index="Noise_Percent",
    columns="Model",
    values="RMSE"
)

print("RMSE UNDER INCREASING SENSOR NOISE")
print("=" * 70)

display(
    rmse_table.style.format("{:.4f}")
)

# ------------------------------------------------------------
# Display percentage degradation
# ------------------------------------------------------------

change_table = noise_results_all.pivot(
    index="Noise_Percent",
    columns="Model",
    values="RMSE_Change_Percent"
)

print("\nRMSE CHANGE RELATIVE TO CLEAN TEST DATA (%)")
print("=" * 70)

display(
    change_table.style.format("{:+.2f}%")
)

# ------------------------------------------------------------
# Full numerical results
# ------------------------------------------------------------

print("\nFULL RESULTS")
print("=" * 70)

display(
    noise_results_all.style.format({
        "Noise_Level": "{:.2f}",
        "Noise_Percent": "{:.0f}%",
        "RMSE": "{:.4f}",
        "Baseline_RMSE": "{:.4f}",
        "RMSE_Change": "{:+.4f}",
        "RMSE_Change_Percent": "{:+.2f}%"
    })
)

In [ ]:
import numpy as np
import pandas as pd
import shap
import time

# ============================================================
# SVR KernelSHAP nsamples sensitivity — setup
# ============================================================

# Fixed background: same size as main SVR SHAP experiment
svr_sens_background = shap.sample(
    X_train[final_features],
    50,
    random_state=42
)

# Small fixed explanation set to keep runtime manageable
svr_sens_test = X_test[final_features].sample(
    n=10,
    random_state=42
)

# Prediction wrapper preserving feature names
def svr_sens_predict(x):
    x_df = pd.DataFrame(
        x,
        columns=final_features
    )
    return svr_model.predict(x_df)

# Create one explainer and reuse it
svr_sens_explainer = shap.KernelExplainer(
    svr_sens_predict,
    svr_sens_background
)

print("KernelSHAP sensitivity setup")
print("=" * 50)
print("Background observations :", len(svr_sens_background))
print("Explanation observations:", len(svr_sens_test))
print("Number of features       :", len(final_features))
print("\nTest row indices:")
print(list(svr_sens_test.index))

In [ ]:
# ============================================================
# KernelSHAP sensitivity test — nsamples = 50
# ============================================================

start_time = time.time()

shap_ns50 = svr_sens_explainer.shap_values(
    svr_sens_test,
    nsamples=50
)

runtime_ns50 = time.time() - start_time

# Mean absolute SHAP importance
importance_ns50 = np.abs(shap_ns50).mean(axis=0)

ranking_ns50 = (
    pd.DataFrame({
        "Feature": final_features,
        "MeanAbsSHAP": importance_ns50
    })
    .sort_values("MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)

ranking_ns50["Rank"] = np.arange(1, len(ranking_ns50) + 1)

print("SVR KernelSHAP — nsamples = 50")
print("=" * 50)
print(f"Runtime: {runtime_ns50:.2f} seconds")
print("\nFeature ranking:")
display(
    ranking_ns50[
        ["Rank", "Feature", "MeanAbsSHAP"]
    ].style.format({
        "MeanAbsSHAP": "{:.6f}"
    })
)

print("\nTop 5:")
print(ranking_ns50["Feature"].head(5).tolist())

In [ ]:
# ============================================================
# KernelSHAP sensitivity test — nsamples = 100
# ============================================================

start_time = time.time()

shap_ns100 = svr_sens_explainer.shap_values(
    svr_sens_test,
    nsamples=100
)

runtime_ns100 = time.time() - start_time

importance_ns100 = np.abs(shap_ns100).mean(axis=0)

ranking_ns100 = (
    pd.DataFrame({
        "Feature": final_features,
        "MeanAbsSHAP": importance_ns100
    })
    .sort_values("MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)

ranking_ns100["Rank"] = np.arange(1, len(ranking_ns100) + 1)

print("SVR KernelSHAP — nsamples = 100")
print("=" * 50)
print(f"Runtime: {runtime_ns100:.2f} seconds")

print("\nFeature ranking:")
display(
    ranking_ns100[
        ["Rank", "Feature", "MeanAbsSHAP"]
    ].style.format({
        "MeanAbsSHAP": "{:.6f}"
    })
)

print("\nTop 5:")
print(ranking_ns100["Feature"].head(5).tolist())

In [ ]:
# ============================================================
# KernelSHAP sensitivity test — nsamples = 200
# ============================================================

start_time = time.time()

shap_ns200 = svr_sens_explainer.shap_values(
    svr_sens_test,
    nsamples=200
)

runtime_ns200 = time.time() - start_time

importance_ns200 = np.abs(shap_ns200).mean(axis=0)

ranking_ns200 = (
    pd.DataFrame({
        "Feature": final_features,
        "MeanAbsSHAP": importance_ns200
    })
    .sort_values("MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)

ranking_ns200["Rank"] = np.arange(1, len(ranking_ns200) + 1)

print("SVR KernelSHAP — nsamples = 200")
print("=" * 50)
print(f"Runtime: {runtime_ns200:.2f} seconds")

print("\nFeature ranking:")
display(
    ranking_ns200[
        ["Rank", "Feature", "MeanAbsSHAP"]
    ].style.format({
        "MeanAbsSHAP": "{:.6f}"
    })
)

print("\nTop 5:")
print(ranking_ns200["Feature"].head(5).tolist())

In [ ]:
from scipy.stats import spearmanr

# ============================================================
# Compare KernelSHAP rankings: 50 vs 100 vs 200 nsamples
# ============================================================

rankings = {
    50: ranking_ns50,
    100: ranking_ns100,
    200: ranking_ns200
}

def get_rank_vector(ranking_df):
    return (
        ranking_df
        .set_index("Feature")["Rank"]
        .reindex(final_features)
        .values
    )

def top_k_set(ranking_df, k=5):
    return set(ranking_df.head(k)["Feature"])

comparison_rows = []

pairs = [
    (50, 100),
    (50, 200),
    (100, 200)
]

for a, b in pairs:

    ranks_a = get_rank_vector(rankings[a])
    ranks_b = get_rank_vector(rankings[b])

    rho, _ = spearmanr(ranks_a, ranks_b)

    top_a = top_k_set(rankings[a], 5)
    top_b = top_k_set(rankings[b], 5)

    intersection = len(top_a & top_b)
    union = len(top_a | top_b)

    jaccard = intersection / union

    comparison_rows.append({
        "Comparison": f"{a} vs {b}",
        "Spearman_rho": rho,
        "Top5_Jaccard": jaccard,
        "Shared_Top5": ", ".join(sorted(top_a & top_b))
    })

sensitivity_comparison = pd.DataFrame(comparison_rows)

print("SVR KernelSHAP nsamples sensitivity")
print("=" * 70)

display(
    sensitivity_comparison.style.format({
        "Spearman_rho": "{:.4f}",
        "Top5_Jaccard": "{:.4f}"
    })
)

print("\nRuntime:")
print(f"nsamples=50 : {runtime_ns50:.2f} s")
print(f"nsamples=100: {runtime_ns100:.2f} s")
print(f"nsamples=200: {runtime_ns200:.2f} s")

In [ ]:
# ============================================================
# KernelSHAP sensitivity test — nsamples = 500
# ============================================================

start_time = time.time()

shap_ns500 = svr_sens_explainer.shap_values(
    svr_sens_test,
    nsamples=500
)

runtime_ns500 = time.time() - start_time

importance_ns500 = np.abs(shap_ns500).mean(axis=0)

ranking_ns500 = (
    pd.DataFrame({
        "Feature": final_features,
        "MeanAbsSHAP": importance_ns500
    })
    .sort_values("MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)

ranking_ns500["Rank"] = np.arange(1, len(ranking_ns500) + 1)

print("SVR KernelSHAP — nsamples = 500")
print("=" * 50)
print(f"Runtime: {runtime_ns500:.2f} seconds")

display(
    ranking_ns500[
        ["Rank", "Feature", "MeanAbsSHAP"]
    ].style.format({
        "MeanAbsSHAP": "{:.6f}"
    })
)

print("\nTop 5:")
print(ranking_ns500["Feature"].head(5).tolist())

In [ ]:
# ============================================================
# FINAL KernelSHAP nsamples sensitivity comparison
# 50 / 100 / 200 / 500
# ============================================================

rankings_final = {
    50: ranking_ns50,
    100: ranking_ns100,
    200: ranking_ns200,
    500: ranking_ns500
}

pairs_final = [
    (50, 100),
    (50, 200),
    (50, 500),
    (100, 200),
    (100, 500),
    (200, 500)
]

comparison_rows = []

for a, b in pairs_final:

    ranks_a = (
        rankings_final[a]
        .set_index("Feature")["Rank"]
        .reindex(final_features)
        .values
    )

    ranks_b = (
        rankings_final[b]
        .set_index("Feature")["Rank"]
        .reindex(final_features)
        .values
    )

    rho, _ = spearmanr(ranks_a, ranks_b)

    top_a = set(rankings_final[a].head(5)["Feature"])
    top_b = set(rankings_final[b].head(5)["Feature"])

    intersection = top_a & top_b
    jaccard = len(intersection) / len(top_a | top_b)

    comparison_rows.append({
        "Comparison": f"{a} vs {b}",
        "Spearman_rho": rho,
        "Top5_Jaccard": jaccard,
        "Shared_Top5": ", ".join(sorted(intersection))
    })

final_nsensitivity = pd.DataFrame(comparison_rows)

print("FINAL SVR KernelSHAP nsamples sensitivity")
print("=" * 75)

display(
    final_nsensitivity.style.format({
        "Spearman_rho": "{:.4f}",
        "Top5_Jaccard": "{:.4f}"
    })
)

print("\nTOP-5 FEATURES AT EACH NSAMPLES")
print("=" * 75)

for n in [50, 100, 200, 500]:
    print(
        f"{n:>3}: "
        f"{rankings_final[n]['Feature'].head(5).tolist()}"
    )

print("\nRUNTIME")
print("=" * 75)
print(f" 50 : {runtime_ns50:.2f} seconds")
print(f"100 : {runtime_ns100:.2f} seconds")
print(f"200 : {runtime_ns200:.2f} seconds")
print(f"500 : {runtime_ns500:.2f} seconds")

In [ ]:
# ============================================================
# CORRELATED-FEATURE ANALYSIS — STEP 1
# Correlation structure of retained predictors
# ============================================================

# Training data only
feature_corr = X_train[final_features].corr(method="pearson")

# Extract unique feature pairs
corr_pairs = []

for i in range(len(final_features)):
    for j in range(i + 1, len(final_features)):

        f1 = final_features[i]
        f2 = final_features[j]

        r = feature_corr.loc[f1, f2]

        corr_pairs.append({
            "Feature_1": f1,
            "Feature_2": f2,
            "Pearson_r": r,
            "Abs_Pearson_r": abs(r)
        })

corr_pairs = (
    pd.DataFrame(corr_pairs)
    .sort_values("Abs_Pearson_r", ascending=False)
    .reset_index(drop=True)
)

print("TOP 20 MOST STRONGLY CORRELATED RETAINED FEATURE PAIRS")
print("=" * 75)

display(
    corr_pairs.head(20).style.format({
        "Pearson_r": "{:.4f}",
        "Abs_Pearson_r": "{:.4f}"
    })
)

print("\nSENSOR_9 / SENSOR_14 CORRELATION")
print("=" * 75)

print(
    f"Pearson correlation: "
    f"{feature_corr.loc['sensor_9', 'sensor_14']:.4f}"
)

In [ ]:
# ============================================================
# CORRELATED-FEATURE ANALYSIS — STEP 2
# Ablation of sensor_9 and sensor_14
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

feature_sets_corr = {
    "All_16": final_features,
    "Without_sensor_9": [
        f for f in final_features if f != "sensor_9"
    ],
    "Without_sensor_14": [
        f for f in final_features if f != "sensor_14"
    ]
}

corr_models = {}
corr_performance = []

for feature_set_name, features in feature_sets_corr.items():

    print(f"\nTraining: {feature_set_name}")
    print("-" * 60)

    # Same frozen RF configuration
    rf_corr = RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )

    # Same frozen XGBoost configuration
    xgb_corr = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    )

    for model_name, model in {
        "RF": rf_corr,
        "XGB": xgb_corr
    }.items():

        model.fit(
            X_train[features],
            y_train
        )

        pred = model.predict(
            X_test[features]
        )

        rmse = np.sqrt(
            mean_squared_error(y_test, pred)
        )

        corr_models[(feature_set_name, model_name)] = model

        corr_performance.append({
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "Num_Features": len(features),
            "RMSE": rmse
        })

corr_performance = pd.DataFrame(corr_performance)

# Baseline for each model
baseline_corr = (
    corr_performance[
        corr_performance["Feature_Set"] == "All_16"
    ]
    .set_index("Model")["RMSE"]
)

corr_performance["RMSE_Change"] = (
    corr_performance.apply(
        lambda row:
        row["RMSE"] - baseline_corr[row["Model"]],
        axis=1
    )
)

corr_performance["RMSE_Change_Percent"] = (
    corr_performance.apply(
        lambda row:
        100 * row["RMSE_Change"] /
        baseline_corr[row["Model"]],
        axis=1
    )
)

print("\nCORRELATED-FEATURE ABLATION — PERFORMANCE")
print("=" * 75)

display(
    corr_performance.style.format({
        "RMSE": "{:.4f}",
        "RMSE_Change": "{:+.4f}",
        "RMSE_Change_Percent": "{:+.2f}%"
    })
)

In [ ]:
# ============================================================
# CORRELATED-FEATURE ANALYSIS — STEP 3
# TreeSHAP attribution redistribution
# ============================================================

import shap
import numpy as np
import pandas as pd

# Same fixed test observations for every configuration
corr_shap_indices = X_test.sample(
    n=1000,
    random_state=42
).index

shap_ablation_results = {}

for feature_set_name, features in feature_sets_corr.items():

    X_explain = X_test.loc[
        corr_shap_indices,
        features
    ]

    for model_name in ["RF", "XGB"]:

        print(
            f"Calculating SHAP: "
            f"{model_name} — {feature_set_name}"
        )

        model = corr_models[
            (feature_set_name, model_name)
        ]

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_explain)

        mean_abs_shap = (
            np.abs(shap_values).mean(axis=0)
        )

        ranking = (
            pd.DataFrame({
                "Feature": features,
                "MeanAbsSHAP": mean_abs_shap
            })
            .sort_values(
                "MeanAbsSHAP",
                ascending=False
            )
            .reset_index(drop=True)
        )

        ranking["Rank"] = (
            np.arange(1, len(ranking) + 1)
        )

        shap_ablation_results[
            (feature_set_name, model_name)
        ] = ranking


# ============================================================
# Focus specifically on sensor_9 and sensor_14
# ============================================================

focus_rows = []

for model_name in ["RF", "XGB"]:

    for feature_set_name in feature_sets_corr.keys():

        ranking = shap_ablation_results[
            (feature_set_name, model_name)
        ]

        for sensor in ["sensor_9", "sensor_14"]:

            if sensor in ranking["Feature"].values:

                row = ranking[
                    ranking["Feature"] == sensor
                ].iloc[0]

                focus_rows.append({
                    "Model": model_name,
                    "Feature_Set": feature_set_name,
                    "Sensor": sensor,
                    "Rank": int(row["Rank"]),
                    "MeanAbsSHAP":
                        row["MeanAbsSHAP"]
                })

focus_table = pd.DataFrame(focus_rows)

print("\nSENSOR_9 / SENSOR_14 SHAP REDISTRIBUTION")
print("=" * 80)

display(
    focus_table.style.format({
        "MeanAbsSHAP": "{:.4f}"
    })
)


# ============================================================
# Top-10 rankings for each experiment
# ============================================================

for model_name in ["RF", "XGB"]:

    print("\n" + "=" * 80)
    print(f"{model_name} TOP-10 SHAP RANKINGS")
    print("=" * 80)

    for feature_set_name in feature_sets_corr.keys():

        print(f"\n{feature_set_name}")

        display(
            shap_ablation_results[
                (feature_set_name, model_name)
            ][
                ["Rank", "Feature", "MeanAbsSHAP"]
            ]
            .head(10)
            .style.format({
                "MeanAbsSHAP": "{:.4f}"
            })
        )

In [ ]:
# Search the notebook variables for likely classifier objects

possible_classifier_objects = []

for name, obj in globals().items():
    name_lower = name.lower()

    if any(word in name_lower for word in [
        "class", "clf", "classifier",
        "failure", "prob"
    ]):
        possible_classifier_objects.append(
            (name, type(obj).__name__)
        )

print("Possible classifier-related objects:")
print("=" * 60)

for name, obj_type in possible_classifier_objects:
    print(f"{name:<40} {obj_type}")

In [ ]:
# ============================================================
# NEAR-FAILURE CLASSIFICATION — STEP 1
# Target definition and data verification
# ============================================================

import numpy as np
import pandas as pd

# Binary target:
# 1 = near failure (RUL < 30 cycles)
# 0 = not near failure (RUL >= 30 cycles)

y_train_cls = (y_train < 30).astype(int)
y_test_cls  = (y_test < 30).astype(int)

# Revised 16-feature matrices
X_train_cls = X_train[final_features].copy()
X_test_cls  = X_test[final_features].copy()

print("NEAR-FAILURE CLASSIFICATION SETUP")
print("=" * 65)

print("Target definition: 1 if RUL < 30 cycles, otherwise 0")
print()

print("Training observations :", len(X_train_cls))
print("Test observations     :", len(X_test_cls))
print("Number of features    :", X_train_cls.shape[1])

print("\nTRAINING CLASS DISTRIBUTION")
print("-" * 65)

train_counts = pd.Series(y_train_cls).value_counts().sort_index()
train_pct = pd.Series(y_train_cls).value_counts(
    normalize=True
).sort_index() * 100

for cls in [0, 1]:
    print(
        f"Class {cls}: "
        f"{train_counts.get(cls, 0):5d} observations "
        f"({train_pct.get(cls, 0):.2f}%)"
    )

print("\nTEST CLASS DISTRIBUTION")
print("-" * 65)

test_counts = pd.Series(y_test_cls).value_counts().sort_index()
test_pct = pd.Series(y_test_cls).value_counts(
    normalize=True
).sort_index() * 100

for cls in [0, 1]:
    print(
        f"Class {cls}: "
        f"{test_counts.get(cls, 0):5d} observations "
        f"({test_pct.get(cls, 0):.2f}%)"
    )

print("\nSanity checks")
print("-" * 65)
print("Training shape:", X_train_cls.shape)
print("Test shape    :", X_test_cls.shape)
print("Missing X train:", X_train_cls.isna().sum().sum())
print("Missing X test :", X_test_cls.isna().sum().sum())

print(
    "Near-failure test observations:",
    int(y_test_cls.sum())
)

In [ ]:
# ============================================================
# NEAR-FAILURE CLASSIFICATION — STEP 2
# Train RF and XGBoost classifiers on revised 16 features
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# Random Forest Classifier
# Same configuration as original thesis experiment
# ------------------------------------------------------------

rf_clf_rev = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_clf_rev.fit(X_train_cls, y_train_cls)

rf_pred = rf_clf_rev.predict(X_test_cls)
rf_prob = rf_clf_rev.predict_proba(X_test_cls)[:, 1]


# ------------------------------------------------------------
# XGBoost Classifier
# Same configuration as original thesis experiment
# ------------------------------------------------------------

xgb_clf_rev = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

xgb_clf_rev.fit(X_train_cls, y_train_cls)

xgb_pred = xgb_clf_rev.predict(X_test_cls)
xgb_prob = xgb_clf_rev.predict_proba(X_test_cls)[:, 1]


# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def classifier_metrics(name, y_true, y_pred, y_prob):

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob)
    }


classification_results = pd.DataFrame([
    classifier_metrics(
        "Random Forest",
        y_test_cls,
        rf_pred,
        rf_prob
    ),

    classifier_metrics(
        "XGBoost",
        y_test_cls,
        xgb_pred,
        xgb_prob
    )
])


print("REVISED 16-FEATURE NEAR-FAILURE CLASSIFICATION")
print("=" * 75)

display(
    classification_results.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "ROC_AUC": "{:.4f}"
    })
)


# ------------------------------------------------------------
# Confusion matrices
# ------------------------------------------------------------

print("\nRANDOM FOREST CONFUSION MATRIX")
print("=" * 50)
print(confusion_matrix(y_test_cls, rf_pred))

print("\nXGBOOST CONFUSION MATRIX")
print("=" * 50)
print(confusion_matrix(y_test_cls, xgb_pred))


# ------------------------------------------------------------
# XGBoost detailed report
# ------------------------------------------------------------

print("\nXGBOOST CLASSIFICATION REPORT")
print("=" * 50)

print(
    classification_report(
        y_test_cls,
        xgb_pred,
        target_names=[
            "RUL >= 30",
            "RUL < 30"
        ],
        digits=4
    )
)

In [ ]:
# ============================================================
# NEAR-FAILURE CLASSIFICATION — STEP 3
# XGBoost probability calibration
# ============================================================

from sklearn.metrics import brier_score_loss, log_loss
from sklearn.calibration import calibration_curve
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Probability-quality metrics
# ------------------------------------------------------------

brier = brier_score_loss(y_test_cls, xgb_prob)
logloss = log_loss(y_test_cls, xgb_prob)

print("XGBOOST PROBABILITY CALIBRATION")
print("=" * 65)

print(f"Brier score : {brier:.4f}")
print(f"Log loss    : {logloss:.4f}")


# ------------------------------------------------------------
# Calibration curve
# Quantile bins give approximately similar numbers of
# observations per bin.
# ------------------------------------------------------------

prob_true, prob_pred = calibration_curve(
    y_test_cls,
    xgb_prob,
    n_bins=10,
    strategy="quantile"
)

calibration_table = pd.DataFrame({
    "Mean_Predicted_Probability": prob_pred,
    "Observed_NearFailure_Rate": prob_true
})

calibration_table["Calibration_Gap"] = (
    calibration_table["Mean_Predicted_Probability"]
    - calibration_table["Observed_NearFailure_Rate"]
)

print("\nCALIBRATION TABLE")
print("=" * 65)

display(
    calibration_table.style.format({
        "Mean_Predicted_Probability": "{:.4f}",
        "Observed_NearFailure_Rate": "{:.4f}",
        "Calibration_Gap": "{:+.4f}"
    })
)


# ------------------------------------------------------------
# Extra summary: mean predicted probability
# ------------------------------------------------------------

print("\nOVERALL CHECK")
print("=" * 65)

print(
    f"Actual near-failure prevalence : "
    f"{y_test_cls.mean():.4f}"
)

print(
    f"Mean predicted probability     : "
    f"{xgb_prob.mean():.4f}"
)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train:", len(y_train))
print("y_test :", len(y_test))
print("Number of final features:", len(final_features))

print("\nFinal features:")
print(final_features)

print("\nTrain engines:", train_data["engine_id"].nunique())
print("Test engines :", test_data["engine_id"].nunique())

In [ ]:
# ============================================================
# Restore final revised XGBoost regression model
# ============================================================

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

xgb_reg_final = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

xgb_reg_final.fit(X_train, y_train)

# Verify reproduction
xgb_pred_check = xgb_reg_final.predict(X_test)

rmse_check = np.sqrt(
    mean_squared_error(y_test, xgb_pred_check)
)

print("Revised XGBoost regression model restored.")
print(f"RMSE = {rmse_check:.4f}")

In [ ]:
# ============================================================
# CASE STUDY REVISION — Candidate Selection
# ============================================================

import numpy as np
import pandas as pd
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. Recreate near-failure target
# ------------------------------------------------------------

y_train_cls = (y_train < 30).astype(int)
y_test_cls  = (y_test < 30).astype(int)

# ------------------------------------------------------------
# 2. Restore revised XGBoost classifier
# ------------------------------------------------------------

xgb_clf_final = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

xgb_clf_final.fit(X_train, y_train_cls)

near_failure_prob = xgb_clf_final.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 3. Revised RUL predictions
# ------------------------------------------------------------

predicted_rul = xgb_reg_final.predict(X_test)

# ------------------------------------------------------------
# 4. Build case-study table
# ------------------------------------------------------------

case_table = pd.DataFrame(index=X_test.index)

case_table["engine_id"] = test_data.loc[
    X_test.index, "engine_id"
]

case_table["cycle"] = test_data.loc[
    X_test.index, "cycle"
]

case_table["actual_RUL"] = y_test

case_table["predicted_RUL"] = predicted_rul

case_table["prediction_error"] = (
    case_table["predicted_RUL"] -
    case_table["actual_RUL"]
)

case_table["absolute_error"] = (
    case_table["prediction_error"].abs()
)

case_table["near_failure_probability"] = near_failure_prob

case_table["actual_near_failure"] = (
    case_table["actual_RUL"] < 30
).astype(int)

# ------------------------------------------------------------
# 5. Candidate groups
# ------------------------------------------------------------

high_rul = (
    case_table[
        case_table["actual_RUL"] >= 80
    ]
    .sort_values("absolute_error")
    .head(10)
)

intermediate_rul = (
    case_table[
        (case_table["actual_RUL"] >= 30) &
        (case_table["actual_RUL"] < 60)
    ]
    .sort_values("absolute_error")
    .head(10)
)

near_failure = (
    case_table[
        case_table["actual_RUL"] < 30
    ]
    .sort_values("absolute_error")
    .head(10)
)

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("HIGH-RUL CANDIDATES (Actual RUL >= 80)")
display(high_rul.round(4))

print("\nINTERMEDIATE-RUL CANDIDATES (30 <= Actual RUL < 60)")
display(intermediate_rul.round(4))

print("\nNEAR-FAILURE CANDIDATES (Actual RUL < 30)")
display(near_failure.round(4))

In [ ]:
# PHM08 / official final-cycle evaluation — environment check

objects_to_check = [
    "train", "test", "rul",
    "test_df", "rul_df",
    "final_features",
    "X_train", "X_test", "y_train", "y_test",
    "lr_model", "rf_state", "xgb_state", "svr_model",
    "lr_final", "rf_final", "xgb_final", "svr_final",
    "xgb_reg_final"
]

print("AVAILABLE OBJECTS")
print("-" * 50)

for name in objects_to_check:
    if name in globals():
        obj = globals()[name]
        try:
            print(f"{name:20s} : {type(obj).__name__:20s} shape={obj.shape}")
        except:
            print(f"{name:20s} : {type(obj).__name__}")

In [ ]:
print("\nFinal features:")
print(final_features if "final_features" in globals() else "final_features NOT FOUND")

In [ ]:
# Verify the exact FD001 files used by this notebook
for label, path in {
    "train_FD001": TRAIN_PATH,
    "test_FD001": TEST_PATH,
    "RUL_FD001": RUL_PATH,
}.items():
    print(f"{label:12s}: {path} | exists={os.path.exists(path)}")

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Load NASA C-MAPSS FD001
# ---------------------------------------------------------

train_path = TRAIN_PATH
test_path  = TEST_PATH
rul_path   = RUL_PATH

# Standard C-MAPSS column structure
columns = (
    ["engine_id", "cycle",
     "op1", "op2", "op3"] +
    [f"s{i}" for i in range(1, 22)]
)

train_fd001 = pd.read_csv(
    train_path,
    sep=r"\s+",
    header=None,
    names=columns
)

test_fd001 = pd.read_csv(
    test_path,
    sep=r"\s+",
    header=None,
    names=columns
)

rul_fd001 = pd.read_csv(
    rul_path,
    sep=r"\s+",
    header=None,
    names=["final_RUL"]
)

# ---------------------------------------------------------
# Construct capped training RUL exactly as in final thesis
# ---------------------------------------------------------

max_cycle = train_fd001.groupby("engine_id")["cycle"].transform("max")

train_fd001["RUL_uncapped"] = max_cycle - train_fd001["cycle"]
train_fd001["RUL"] = train_fd001["RUL_uncapped"].clip(upper=125)

# ---------------------------------------------------------
# Final 16 retained features
# ---------------------------------------------------------

final_features = [
    "op1", "op2",
    "s2", "s3", "s4",
    "s7", "s8", "s9",
    "s11", "s12", "s13", "s14", "s15",
    "s17", "s20", "s21"
]

# ---------------------------------------------------------
# Basic verification
# ---------------------------------------------------------

print("TRAIN:", train_fd001.shape)
print("TEST :", test_fd001.shape)
print("RUL  :", rul_fd001.shape)

print("\nNumber of engines:")
print("Train engines:", train_fd001["engine_id"].nunique())
print("Test engines :", test_fd001["engine_id"].nunique())

print("\nTraining RUL check:")
print(train_fd001["RUL"].describe())

print("\nFinal features:")
print(final_features)
print("Number of features:", len(final_features))

print("\nMissing values:")
print("Train:", train_fd001[final_features].isna().sum().sum())
print("Test :", test_fd001[final_features].isna().sum().sum())

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# ---------------------------------------------------------
# Training data — all 100 NASA training engines
# ---------------------------------------------------------

X_train_official = train_fd001[final_features]
y_train_official = train_fd001["RUL"]

print("Training data:", X_train_official.shape)
print("Training engines:", train_fd001["engine_id"].nunique())

# ---------------------------------------------------------
# 1. Linear Regression
# ---------------------------------------------------------

lr_official = LinearRegression()

# ---------------------------------------------------------
# 2. Random Forest — FINAL thesis configuration
# ---------------------------------------------------------

rf_official = RandomForestRegressor(
    n_estimators=150,
    max_depth=15,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# ---------------------------------------------------------
# 3. XGBoost — FINAL thesis configuration
# ---------------------------------------------------------

xgb_official = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ---------------------------------------------------------
# 4. SVR — FINAL thesis configuration
# ---------------------------------------------------------

svr_official = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",
        C=100,
        epsilon=0.1,
        gamma="scale"
    ))
])

# ---------------------------------------------------------
# Train
# ---------------------------------------------------------

models_official = {
    "Linear Regression": lr_official,
    "Random Forest": rf_official,
    "XGBoost": xgb_official,
    "SVR": svr_official
}

for name, model in models_official.items():
    print(f"Training {name}...")
    model.fit(X_train_official, y_train_official)
    print(f"{name}: DONE")

print("\nAll four final models trained successfully.")

In [ ]:
# ---------------------------------------------------------
# Official FD001 final-cycle evaluation dataset
# ---------------------------------------------------------

# Take the last observed cycle for each of the 100 test engines
test_final = (
    test_fd001
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id", as_index=False)
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

# NASA-provided RUL at the end of each test trajectory
y_final = rul_fd001["final_RUL"].to_numpy()

# Features for prediction
X_final = test_final[final_features]

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("Final-cycle feature matrix:", X_final.shape)
print("NASA RUL labels:", y_final.shape)

print("\nNumber of engines:", test_final["engine_id"].nunique())

print("\nEngine ID range:")
print(test_final["engine_id"].min(), "to", test_final["engine_id"].max())

print("\nFinal observed cycle statistics:")
print(test_final["cycle"].describe())

print("\nNASA final RUL statistics:")
print(pd.Series(y_final).describe())

print("\nFirst 10 engine endpoints:")
check_table = pd.DataFrame({
    "engine_id": test_final["engine_id"],
    "last_observed_cycle": test_final["cycle"],
    "NASA_final_RUL": y_final
})

print(check_table.head(10).to_string(index=False))

# Critical assertions
assert len(test_final) == 100
assert len(y_final) == 100
assert test_final["engine_id"].nunique() == 100
assert list(test_final["engine_id"]) == list(range(1, 101))

print("\n✓ Final-cycle official test dataset verified successfully.")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# PHM08 asymmetric scoring function
#
# Error definition:
# d = predicted RUL - true RUL
#
# d < 0  -> RUL underestimation
# d >= 0 -> RUL overestimation
# ---------------------------------------------------------

def phm08_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    d = y_pred - y_true

    penalties = np.where(
        d < 0,
        np.exp(-d / 13.0) - 1.0,
        np.exp(d / 10.0) - 1.0
    )

    return penalties.sum(), penalties


# ---------------------------------------------------------
# Evaluate all four models
# ---------------------------------------------------------

results = []
prediction_table = pd.DataFrame({
    "engine_id": test_final["engine_id"].to_numpy(),
    "true_RUL": y_final
})

for name, model in models_official.items():

    pred = model.predict(X_final)

    # Store predictions
    prediction_table[name] = pred

    # Standard regression metrics
    rmse = np.sqrt(mean_squared_error(y_final, pred))
    mae = mean_absolute_error(y_final, pred)
    r2 = r2_score(y_final, pred)

    # PHM08
    total_phm, penalties = phm08_score(y_final, pred)

    # Signed prediction error
    error = pred - y_final

    n_over = np.sum(error > 0)
    n_under = np.sum(error < 0)
    n_exact = np.sum(np.isclose(error, 0))

    mean_error = np.mean(error)

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "PHM08_Total": total_phm,
        "PHM08_Mean_per_Engine": np.mean(penalties),
        "Overestimates": n_over,
        "Underestimates": n_under,
        "Exact": n_exact,
        "Mean_Signed_Error": mean_error
    })


results_df = pd.DataFrame(results)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("=" * 105)
print("OFFICIAL FD001 FINAL-CYCLE EVALUATION")
print("=" * 105)

print(
    results_df.to_string(
        index=False,
        formatters={
            "RMSE": "{:.4f}".format,
            "MAE": "{:.4f}".format,
            "R2": "{:.4f}".format,
            "PHM08_Total": "{:.4f}".format,
            "PHM08_Mean_per_Engine": "{:.4f}".format,
            "Mean_Signed_Error": "{:.4f}".format
        }
    )
)

print("\nLower RMSE, MAE and PHM08 scores are better.")
print("Positive mean signed error = average RUL overestimation.")
print("Negative mean signed error = average RUL underestimation.")

In [ ]:
# ---------------------------------------------------------
# Diagnose PHM08 score:
# Which engines contribute the largest asymmetric penalties?
# ---------------------------------------------------------

diagnostic_rows = []

for name, model in models_official.items():

    pred = model.predict(X_final)
    error = pred - y_final

    # PHM08 penalty per engine
    penalty = np.where(
        error < 0,
        np.exp(-error / 13.0) - 1.0,
        np.exp(error / 10.0) - 1.0
    )

    for i in range(len(y_final)):
        diagnostic_rows.append({
            "Model": name,
            "engine_id": int(test_final.iloc[i]["engine_id"]),
            "true_RUL": float(y_final[i]),
            "predicted_RUL": float(pred[i]),
            "signed_error": float(error[i]),
            "error_type": "Overestimate" if error[i] > 0 else "Underestimate",
            "PHM08_penalty": float(penalty[i])
        })

diagnostics = pd.DataFrame(diagnostic_rows)

# ---------------------------------------------------------
# Show five largest PHM08 penalties for each model
# ---------------------------------------------------------

for name in models_official.keys():

    temp = (
        diagnostics[diagnostics["Model"] == name]
        .sort_values("PHM08_penalty", ascending=False)
        .head(5)
    )

    print("\n" + "=" * 85)
    print(name.upper())
    print("Five largest PHM08 penalties")
    print("=" * 85)

    print(
        temp[
            [
                "engine_id",
                "true_RUL",
                "predicted_RUL",
                "signed_error",
                "error_type",
                "PHM08_penalty"
            ]
        ].to_string(
            index=False,
            formatters={
                "true_RUL": "{:.2f}".format,
                "predicted_RUL": "{:.2f}".format,
                "signed_error": "{:+.2f}".format,
                "PHM08_penalty": "{:.2f}".format
            }
        )
    )

# ---------------------------------------------------------
# Concentration of PHM08 score
# ---------------------------------------------------------

print("\n" + "=" * 85)
print("PHM08 SCORE CONCENTRATION")
print("=" * 85)

for name in models_official.keys():

    temp = (
        diagnostics[diagnostics["Model"] == name]
        .sort_values("PHM08_penalty", ascending=False)
    )

    total = temp["PHM08_penalty"].sum()

    top1 = temp.head(1)["PHM08_penalty"].sum() / total * 100
    top5 = temp.head(5)["PHM08_penalty"].sum() / total * 100
    top10 = temp.head(10)["PHM08_penalty"].sum() / total * 100

    print(
        f"{name:20s} | "
        f"Top 1: {top1:6.2f}% | "
        f"Top 5: {top5:6.2f}% | "
        f"Top 10: {top10:6.2f}%"
    )

In [ ]:
# ---------------------------------------------------------
# Effect of NASA final RUL values above training cap = 125
# ---------------------------------------------------------

above_cap = y_final > 125

print("Official endpoints with RUL > 125:", above_cap.sum())
print("Official endpoints with RUL <= 125:", (~above_cap).sum())

print("\nRUL values above 125:")
print(y_final[above_cap])

print("\nEngines with official RUL > 125:")
print(
    pd.DataFrame({
        "engine_id": test_final.loc[above_cap, "engine_id"].to_numpy(),
        "NASA_RUL": y_final[above_cap]
    }).to_string(index=False)
)

print("\n" + "=" * 90)
print("PREDICTIONS FOR ABOVE-CAP ENGINES")
print("=" * 90)

cap_check = pd.DataFrame({
    "engine_id": test_final.loc[above_cap, "engine_id"].to_numpy(),
    "true_RUL": y_final[above_cap]
})

for name, model in models_official.items():
    cap_check[name] = model.predict(X_final.loc[above_cap])

print(
    cap_check.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

In [ ]:
# ---------------------------------------------------------
# Sensitivity analysis:
# Official final-cycle engines whose NASA RUL <= training cap
# ---------------------------------------------------------

within_cap = y_final <= 125

X_final_125 = X_final.loc[within_cap]
y_final_125 = y_final[within_cap]

results_125 = []

for name, model in models_official.items():

    pred = model.predict(X_final_125)

    rmse = np.sqrt(mean_squared_error(y_final_125, pred))
    mae = mean_absolute_error(y_final_125, pred)
    r2 = r2_score(y_final_125, pred)

    phm_total, penalties = phm08_score(y_final_125, pred)

    error = pred - y_final_125

    results_125.append({
        "Model": name,
        "N": len(y_final_125),
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "PHM08_Total": phm_total,
        "Overestimates": np.sum(error > 0),
        "Underestimates": np.sum(error < 0),
        "Mean_Signed_Error": np.mean(error)
    })

results_125_df = pd.DataFrame(results_125)

print("=" * 100)
print("SENSITIVITY ANALYSIS — OFFICIAL ENDPOINTS WITH NASA RUL <= 125")
print("=" * 100)

print(
    results_125_df.to_string(
        index=False,
        formatters={
            "RMSE": "{:.4f}".format,
            "MAE": "{:.4f}".format,
            "R2": "{:.4f}".format,
            "PHM08_Total": "{:.4f}".format,
            "Mean_Signed_Error": "{:.4f}".format
        }
    )
)